# Data Microstructure

# Setup

## Imports

In [ ]:
import os
import warnings
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm

from thesis_project import (
    config,
)
from thesis_project.collectors import (
    run_collectors_double,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "data-microstructure"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    # OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    # OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    # OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

## Utilities

### Plotting

# Main

## Common

In [ ]:
# Common utilities
from thesis_project.collectors import (
    run_collectors_single,
)

# Common data
cf_df = pd.read_csv(
    LOB_DIR / "daily_cf.csv", parse_dates=["Date"], date_format="%Y-%m-%d", index_col="Date"
)

ctd_meta_df = pd.read_csv(
    LOB_DIR / "ctd_metadata.csv", parse_dates=["Issue Date", "Maturity Date", "First Coupon Date"]
)

fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

if TICKER == "fbtp":
    START_YEAR = config.FBTP_START_YEAR
    END_YEAR = config.FBTP_END_YEAR

if TICKER == "fbts":
    START_YEAR = config.FBTS_START_YEAR
    END_YEAR = config.FBTS_END_YEAR

## Bid/Ask Price

In [ ]:
from thesis_project.collectors import compute_bid_ask_price

LOBS = {
    "futures": LOB_FUT_DIR,
    "ctd": LOB_CTD_DIR,
}

key = "Price"
extractor = compute_bid_ask_price
extractors = {key: extractor}
lob_lvls = range(1, config.LEVELS + 1)
sides = ["bid", "ask"]

results = defaultdict(lambda: defaultdict(dict))

for lob_type, lob_dir in tqdm(LOBS.items(), desc="LOB types", unit="LOB type"):
    if lob_type == "futures":
        start_time = config.FUT_MRKT_OPEN
        end_time = config.FUT_MRKT_CLOSE
    if lob_type == "ctd":
        start_time = config.CTD_MRKT_OPEN
        end_time = config.CTD_MRKT_CLOSE

    for lob_lvl in tqdm(lob_lvls, desc="LOB levels", unit="LOB level", leave=False):
        for side in sides:
            context = {
                "LOB level": lob_lvl,
                "Side": side,
            }
            results[lob_type][lob_lvl][side] = run_collectors_single(
                lob_dir,
                lob_type,
                extractors=extractors,
                start_year=START_YEAR,
                end_year=END_YEAR,
                start_time=start_time,
                end_time=end_time,
                context=context,
            )

In [ ]:
name_template = "{side}_price_lvl_{lvl}"
freq = "1s"
subdir = "price"

for lob_type in LOBS.keys():
    for lob_lvl in lob_lvls:
        for side in sides:
            data = results[lob_type][lob_lvl][side][key]

            start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
            end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

            name = name_template.format(side=side, lvl=lob_lvl)
            prefix = "fut" if lob_type == "futures" else "ctd"
            name = f"{prefix}_{name}"
            filename = config.SERIES_FILENAME_TEMPLATE.format(
                series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
            )

            # print(filename)
            # print(f"Data: from {data.index[0]} to {data.index[-1]}\n")
            # display(data.head())

            dir = OUT_INT_DIR / subdir
            os.makedirs(OUT_INT_DIR / dir, exist_ok=True)
            data.to_parquet(OUT_INT_DIR / subdir / filename)

## Bid/Ask Size

In [ ]:
from thesis_project.collectors import compute_bid_ask_size

LOBS = {
    "futures": LOB_FUT_DIR,
    "ctd": LOB_CTD_DIR,
}

key = "Size"
extractor = compute_bid_ask_size
extractors = {key: extractor}
lob_lvls = range(1, config.LEVELS + 1)
sides = ["bid", "ask"]

results = defaultdict(lambda: defaultdict(dict))

for lob_type, lob_dir in tqdm(LOBS.items(), desc="LOB types", unit="LOB type"):
    if lob_type == "futures":
        start_time = config.FUT_MRKT_OPEN
        end_time = config.FUT_MRKT_CLOSE
    if lob_type == "ctd":
        start_time = config.CTD_MRKT_OPEN
        end_time = config.CTD_MRKT_CLOSE

    for lob_lvl in tqdm(lob_lvls, desc="LOB levels", unit="LOB level", leave=False):
        for side in sides:
            context = {
                "LOB level": lob_lvl,
                "Side": side,
            }
            results[lob_type][lob_lvl][side] = run_collectors_single(
                lob_dir,
                lob_type,
                extractors=extractors,
                start_year=START_YEAR,
                end_year=END_YEAR,
                start_time=start_time,
                end_time=end_time,
                context=context,
            )

In [ ]:
name_template = "{side}_size_lvl_{lvl}"
freq = "1s"
subdir = "size"

for lob_type in LOBS.keys():
    for lob_lvl in lob_lvls:
        for side in sides:
            data = results[lob_type][lob_lvl][side][key]

            start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
            end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

            name = name_template.format(side=side, lvl=lob_lvl)
            prefix = "fut" if lob_type == "futures" else "ctd"
            name = f"{prefix}_{name}"
            filename = config.SERIES_FILENAME_TEMPLATE.format(
                series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
            )

            # print(filename)
            # print(f"Data: from {data.index[0]} to {data.index[-1]}\n")
            # display(data.head())

            dir = OUT_INT_DIR / subdir
            os.makedirs(OUT_INT_DIR / dir, exist_ok=True)
            data.to_parquet(OUT_INT_DIR / subdir / filename)

## Mid Price

In [ ]:
from thesis_project.collectors import compute_mid_price

LOBS = {
    "futures": LOB_FUT_DIR,
    "ctd": LOB_CTD_DIR,
}

key = "Mid"
extractor = compute_mid_price
extractors = {key: extractor}
context = {
    "LOB level": 1,
}

results = {}
for lob_type, lob_dir in LOBS.items():
    if lob_type == "futures":
        start_time = config.FUT_MRKT_OPEN
        end_time = config.FUT_MRKT_CLOSE
    if lob_type == "ctd":
        start_time = config.CTD_MRKT_OPEN
        end_time = config.CTD_MRKT_CLOSE
    results[lob_type] = run_collectors_single(
        lob_dir,
        lob_type,
        extractors=extractors,
        start_year=START_YEAR,
        end_year=END_YEAR,
        start_time=start_time,
        end_time=end_time,
        context=context,
    )

In [ ]:
name_template = "mid_price"
freq = "1s"
subdir = name.replace("_", "-")

for lob_type in LOBS.keys():
    data = results[lob_type][key]

    start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
    end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

    prefix = "fut" if lob_type == "futures" else "ctd"
    name = f"{prefix}_{name_template}"
    filename = config.SERIES_FILENAME_TEMPLATE.format(
        series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
    )

    # print(filename)
    # display(data.head())

    # dir = OUT_INT_DIR / subdir
    # os.makedirs(OUT_INT_DIR / dir, exist_ok=True)
    # data.to_parquet(OUT_INT_DIR / subdir / filename)

## Spread

In [ ]:
from thesis_project.collectors import compute_spread

LOBS = {
    "futures": LOB_FUT_DIR,
    "ctd": LOB_CTD_DIR,
}

key = "Spread"
extractor = compute_spread
extractors = {key: extractor}

context = {
    "LOB level": 1,
}

results = {}
for lob_type, lob_dir in LOBS.items():
    if lob_type == "futures":
        start_time = config.FUT_MRKT_OPEN
        end_time = config.FUT_MRKT_CLOSE
    if lob_type == "ctd":
        start_time = config.CTD_MRKT_OPEN
        end_time = config.CTD_MRKT_CLOSE
    results[lob_type] = run_collectors_single(
        lob_dir,
        lob_type,
        extractors=extractors,
        start_year=START_YEAR,
        end_year=END_YEAR,
        start_time=start_time,
        end_time=end_time,
        context=context,
    )

In [ ]:
name_template = "spread"
freq = "1s"
subdir = name

for lob_type in LOBS.keys():
    data = results[lob_type][key]

    start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
    end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

    prefix = "fut" if lob_type == "futures" else "ctd"
    name = f"{prefix}_{name_template}"
    filename = config.SERIES_FILENAME_TEMPLATE.format(
        series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
    )

    # print(filename)
    # display(data.head())

    # dir = OUT_INT_DIR / subdir
    # os.makedirs(OUT_INT_DIR / dir, exist_ok=True)
    # data.to_parquet(OUT_INT_DIR / subdir / filename)

## Micro Price

In [ ]:
from thesis_project.collectors import compute_micro_price

LOBS = {
    "futures": LOB_FUT_DIR,
    "ctd": LOB_CTD_DIR,
}

key = "Micro"
extractor = compute_micro_price
extractors = {key: extractor}
context = {
    "LOB level": 1,
}

results = {}
for lob_type, lob_dir in LOBS.items():
    if lob_type == "futures":
        start_time = config.FUT_MRKT_OPEN
        end_time = config.FUT_MRKT_CLOSE
    if lob_type == "ctd":
        start_time = config.CTD_MRKT_OPEN
        end_time = config.CTD_MRKT_CLOSE
    results[lob_type] = run_collectors_single(
        lob_dir,
        lob_type,
        extractors=extractors,
        start_year=START_YEAR,
        end_year=END_YEAR,
        start_time=start_time,
        end_time=end_time,
        context=context,
    )

In [ ]:
name_template = "micro_price"
freq = "1s"
subdir = name.replace("_", "-")

for lob_type in LOBS.keys():
    data = results[lob_type][key]

    start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
    end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

    prefix = "fut" if lob_type == "futures" else "ctd"
    name = f"{prefix}_{name_template}"
    filename = config.SERIES_FILENAME_TEMPLATE.format(
        series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
    )

    # print(filename)
    # display(data.head())

    dir = OUT_INT_DIR / subdir
    os.makedirs(OUT_INT_DIR / dir, exist_ok=True)
    data.to_parquet(OUT_INT_DIR / subdir / filename)

## Order Book Imbalance

In [ ]:
from thesis_project.collectors import compute_obi

LOBS = {
    "futures": LOB_FUT_DIR,
    "ctd": LOB_CTD_DIR,
}

key = "OBI"
extractor = compute_obi
extractors = {key: extractor}
context = {
    "LOB level range": [1],
    "Scale": True,
}

results = {}
for lob_type, lob_dir in LOBS.items():
    if lob_type == "futures":
        start_time = config.FUT_MRKT_OPEN
        end_time = config.FUT_MRKT_CLOSE
    if lob_type == "ctd":
        start_time = config.CTD_MRKT_OPEN
        end_time = config.CTD_MRKT_CLOSE
    results[lob_type] = run_collectors_single(
        lob_dir,
        lob_type,
        extractors=extractors,
        start_year=START_YEAR,
        end_year=END_YEAR,
        start_time=start_time,
        end_time=end_time,
        context=context,
    )

In [ ]:
name_template = "obi_{lvls_range}"
freq = "1s"
subdir = "obi"

for lob_type in LOBS.keys():
    data = results[lob_type][key]

    start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
    end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

    lvl_range = context["LOB level range"]
    if len(lvl_range) == 1:
        name = name_template.format(lvls_range=f"lvl_{lvl_range[0]}")
    else:
        start_lvl = context["LOB level range"][0]
        end_lvl = context["LOB level range"][-1]
        name = name_template.format(lvls_range=f"lvls_{start_lvl}_{end_lvl}")
    if context.get("Scale", False):
        name += "_scaled"
    prefix = "fut" if lob_type == "futures" else "ctd"
    name = f"{prefix}_{name}"
    filename = config.SERIES_FILENAME_TEMPLATE.format(
        series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
    )

    # print(filename)
    # print(f"Data: from {data.index[0]} to {data.index[-1]}\n")
    # display(data.head())

    dir = OUT_INT_DIR / subdir
    os.makedirs(OUT_INT_DIR / dir, exist_ok=True)
    data.to_parquet(OUT_INT_DIR / subdir / filename)

## Order Flow Imbalance

In [ ]:
from thesis_project.collectors import compute_ofi

LOBS = {
    "futures": LOB_FUT_DIR,
    "ctd": LOB_CTD_DIR,
}

key = "OFI"
extractor = compute_ofi
extractors = {key: extractor}
context = {
    "LOB level": 1,
}

results = {}
for lob_type, lob_dir in LOBS.items():
    if lob_type == "futures":
        start_time = config.FUT_MRKT_OPEN
        end_time = config.FUT_MRKT_CLOSE
    if lob_type == "ctd":
        start_time = config.CTD_MRKT_OPEN
        end_time = config.CTD_MRKT_CLOSE
    results[lob_type] = run_collectors_single(
        lob_dir,
        lob_type,
        extractors=extractors,
        start_year=START_YEAR,
        end_year=END_YEAR,
        start_time=start_time,
        end_time=end_time,
        context=context,
    )

In [ ]:
name_template = "ofi_lvl_{lvl}"
freq = "1s"
subdir = "ofi"

for lob_type in LOBS.keys():
    data = results[lob_type][key]

    start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
    end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

    lvl = context["LOB level"]
    name = name_template.format(lvl=lvl)
    filename = config.SERIES_FILENAME_TEMPLATE.format(
        series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
    )

    # print(filename)
    # print(f"Data: from {data.index[0]} to {data.index[-1]}\n")
    # display(data.head())

    dir = OUT_INT_DIR / subdir
    os.makedirs(OUT_INT_DIR / dir, exist_ok=True)
    data.to_parquet(OUT_INT_DIR / subdir / filename)

## Gross Basis

In [ ]:
from thesis_project.collectors import compute_gross_basis

context = {
    "CF": cf_df,
}

results = run_collectors_double(
    LOB_FUT_DIR,
    LOB_CTD_DIR,
    extractors={"Basis": compute_gross_basis},
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
    context=context,
)

In [ ]:
data = results["Basis"]

start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

name = "basis"
freq = "1s"
filename = config.SERIES_FILENAME_TEMPLATE.format(
    series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
)

subdir = "basis"
# data.to_parquet(OUT_INT_DIR / subdir / filename)

## Implied Repo Rate

In [ ]:
from thesis_project.collectors import compute_implied_repo

context = {
    "CF": cf_df,
    "CTD Metadata": ctd_meta_df,
    "FUT Metadata": fut_meta_df,
    "Frequency": "1s",
    "Notional": 100.0,
}

results = run_collectors_double(
    LOB_FUT_DIR,
    LOB_CTD_DIR,
    extractors={"IRR": compute_implied_repo},
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
    context=context,
)

In [ ]:
data = results["IRR"]

start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

name = "irr"
freq = context["Frequency"]
filename = config.SERIES_FILENAME_TEMPLATE.format(
    series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
)

subdir = "irr"
data.to_parquet(OUT_INT_DIR / subdir / filename)